# BBSure tutorial

This notebook goes through the whole pipeline on one data set of the validation:
a simulated SO SAT observation with **PySM d12s5** foregrounds (multi-layer dust,
synchrotron with curvature), CMB with **r = 0.0111** and SO pessimistic-goal noise
(seed 7047).

1. **Producing the data**: mask, Gaussian foreground model, covariance sims, fiducial,
   maps and spectra. These steps need PySM, NaMaster and several CPU hours, so the
   cells are commented out: the spectra they produce are shipped in
   `data/examples/d12s5_s7047/` (BB only).
2. **The example spectra.**
3. **The decision tree and the profile scheme** on the example: configuration and run
   (a few hours on a laptop; the fits are cached, so a second run is fast).
4. **The results**: the decisions of the tree, the Bayesian posterior of r, the profile
   likelihood of r.

The two flowcharts:

| Bayesian tree | profile scheme |
|---|---|
| ![tree](../docs/flowcharts/decision_tree_v3_profile.png) | ![profile](../docs/flowcharts/profile_scheme_nersc.png) |

## 0. Setup

In [ ]:
import json
import os
import subprocess

import numpy as np
import matplotlib.pyplot as plt
import sacc
import yaml

import bbsure
from bbsure.paths import DATA_DIR

REPO = os.path.dirname(os.path.dirname(os.path.abspath(bbsure.__file__)))
WORK = os.path.join(REPO, 'notebooks', 'tutorial_run')   # output of this notebook
EXAMPLE = os.path.join(DATA_DIR, 'examples', 'd12s5_s7047')
TREE = os.path.join(WORK, 'tree')        # runs_dir of the tree
os.makedirs(WORK, exist_ok=True)

import bbpower   # the BBPower fork, branch Bsure, must be importable
%matplotlib inline
print('BBSure  :', REPO)
print('BBPower :', os.path.dirname(os.path.dirname(bbpower.__file__)))
print('data    :', DATA_DIR)

## 1. Producing the data (commented out)

All the commands are run from the root of the repository. Remove the `#` to run them.

### 1a. Analysis mask

Hits-weighted mask with a C2 apodization of 7 degrees, from the SO SAT hits map in
galactic coordinates (`data/masks/norm_nHits_SA_35FOV_G_nside128.fits`). The result,
`mask_hits_C2_7deg_nside128.fits.gz`, is already in `data/masks/`.

In [ ]:
# !cd {REPO} && python scripts/make_mask.py --apo 7 --hits norm_nHits_SA_35FOV_G_nside128.fits

### 1b. Gaussian foreground model for the covariance

The covariance comes from Gaussian simulations whose foregrounds match PySM **d1s1** in
the mask: power-law $D_\ell$ of dust and synchrotron, mean spectral indices weighted by
the polarized intensity. The fit (PySM at nside 512, harmonic downgrade to 128) gives
`configs/fg_params_d1s1_hits_n128.yml`.

In [ ]:
# !cd {REPO} && python scripts/fit_fg_spectra.py --model d1s1 \
#     --mask mask_hits_C2_7deg_nside128 --nside 128 --nside-in 512 \
#     --edges bpw_edges_nside128.txt --lmin 30 --lmax 250 \
#     --out configs/fg_params_d1s1_hits_n128.yml

### 1c. Covariance sims and their spectra

500 Gaussian sims (CMB with r = 0, foregrounds of 1b, noise following the hits, no E
modes in the signal), then their power spectra with BBPowerSpecter. The spectra of the
sims are computed once (`--cov-maps`) and then reused for every data set
(`--cov-cells`). About 1 CPU minute per sim.

In [ ]:
# !cd {REPO} && bash scripts/generate_sims_cov.sh --nside 128 --sims-dir sims_cov_n128 \
#     --fg-params configs/fg_params_d1s1_hits_n128.yml \
#     --hits data/masks/norm_nHits_SA_35FOV_G_nside128.fits \
#     --noise pessimistic_goal --no-kludge --no-ee --nproc 8
#
# # spectra of the sims (any map set as the "data" of this run, e.g. the first sim)
# !cd {REPO} && bash scripts/run_spectra.sh --maps sims_cov_n128/s5002 --out cov_n128 \
#     --cov-maps sims_cov_n128/sims_list.txt

### 1d. Fiducial spectra of the Hamimeche-Lewis likelihood

The BBPower model (r = 0, power-law foregrounds, no moments) at the parameters of the
covariance sims, i.e. the mean of the sims: `data/cls_fid_d1s1_hits_pg_n128.fits`.

In [ ]:
# !cd {REPO} && python scripts/make_fiducial.py \
#     --fg-params configs/fg_params_d1s1_hits_n128.yml \
#     --base-config configs/base_config_tree.yml --cells cov_n128 \
#     --out data/cls_fid_d1s1_hits_pg_n128.fits

### 1e. The data: maps and spectra

Maps of the data set (PySM d12s5 at nside 512, harmonic downgrade to 128), then the
data vector with the covariance of 1c. This produces the files in
`data/examples/d12s5_s7047/` (there reduced to BB only).

In [ ]:
# !cd {REPO} && python scripts/generate_maps.py --seed 7047 --nside 128 --r 0.01113176 \
#     --pysm d12s5 --pysm-nside-in 512 --no-ee --noise pessimistic_goal --no-kludge \
#     --hits-noise data/masks/norm_nHits_SA_35FOV_G_nside128.fits \
#     --output-dir maps_d12s5_s7047
#
# !cd {REPO} && bash scripts/run_spectra.sh --maps maps_d12s5_s7047 --out spectra_d12s5_s7047 \
#     --cov-cells cov_n128/cells_all_sims.txt

## 2. The example spectra

The BB spectra between the 6 SO SAT bands (`cells_coadded.fits`: cross-split spectra,
without noise bias), with the error bars from the covariance of the sims. The noise
spectra (`cells_noise.fits`) enter the Hamimeche-Lewis likelihood. The tree fits
$30 \le \ell \le 250$.

In [ ]:
s = sacc.Sacc.load_fits(os.path.join(EXAMPLE, 'cells_coadded.fits'))
bands = sorted(s.tracers)
NAMES = dict(zip(bands, ['LF1', 'LF2', 'MF1', 'MF2', 'UHF1', 'UHF2']))
err = np.sqrt(np.diag(s.covariance.covmat))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, pairs, title in [(axes[0], [(b, b) for b in bands], 'auto-spectra'),
                         (axes[1], [(bands[i], bands[j]) for i in range(6)
                                    for j in range(i + 1, 6) if j == i + 1],
                          'neighbouring cross-spectra')]:
    for p in pairs:
        ell, cl, ind = s.get_ell_cl('cl_bb', *p, return_ind=True)
        ok = (ell >= 30) & (ell <= 250)
        ax.errorbar(ell[ok], cl[ok], err[ind][ok], fmt='o', ms=3,
                    label=' x '.join(NAMES[t] for t in p))
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=1e-4)
    ax.set_xlabel(r'$\ell$'); ax.set_title(f'BB {title}')
    ax.legend(fontsize=8)
axes[0].set_ylabel(r'$D_\ell^{BB}$ [$\mu$K$^2$]')
plt.show()

## 3. Configuration of the tree

The tree config sets the thresholds of the decisions, the moment groups tested, the
PolyChord and emcee settings and the profile scheme. We start from the template of the
validation and point it to the example spectra; PolyChord runs with MPI on half of the
cores of this machine.

In [ ]:
tpl = open(os.path.join(REPO, 'configs', 'tree_config_template.yml')).read()
txt = (tpl.replace('@REPO@', REPO).replace('@DATA@', DATA_DIR)
          .replace('@RUN@/output', EXAMPLE).replace('@RUN@', WORK))
tc = yaml.safe_load(txt)
tc['polychord']['mpi_procs'] = max(1, os.cpu_count() // 2)
tc['polychord'].pop('mpi_launcher')              # local mpirun instead of srun
cfg_path = os.path.join(WORK, 'tree_config.yml')
yaml.safe_dump(tc, open(cfg_path, 'w'), sort_keys=False)

print(yaml.safe_dump({k: tc[k] for k in ('paths', 'thresholds', 'moments', 'run')},
                     sort_keys=False))

The moments and their priors are in the BBPower config, `configs/base_config_tree.yml`:
for each group (`d`, `s`, `dT`, `d2`, `s2`, `ds1`, `ds2`) the spectra of the moment maps,
$A\,(\ell/80)^\alpha$. The tree switches them on and off.

In [ ]:
from bbsure.models import MOMENT_GROUPS
for g, G in MOMENT_GROUPS.items():
    print(f"{g:4s} {G['comp']}: {', '.join('A_%s_%s' % (G['pref'], sp) for sp in G['spectra'])}"
          f"   (free with it: {', '.join(G['free'])})")

## 4. Run: decision tree + profile scheme

`scripts/run_tree.py` runs the tree (ML fits, PolyChord evidences, emcee screening of the
residuals) and then, with `run: profile_scheme: true`, the profile scheme on the concluded
tree. On a laptop this takes a few hours; every BBCompSep run is cached in `WORK`
(`config.md5`), so an interrupted run restarts from where it stopped and a second run of
this cell is fast. The log is written in `tutorial_run/run.log`, the decisions in
`tutorial_run/tree/decision_log.txt`.

In [ ]:
%%time
env = dict(os.environ, OMP_NUM_THREADS='1',
           PROFILE_NPROC=str(max(1, os.cpu_count() // 2)),
           BBPOWER_NPROC=str(max(1, os.cpu_count() // 2)))
with open(os.path.join(WORK, 'run.log'), 'w') as log:
    ret = subprocess.run(['python', os.path.join(REPO, 'scripts', 'run_tree.py'),
                          '--config', cfg_path], stdout=log,
                         stderr=subprocess.STDOUT, env=env)
print('exit code', ret.returncode)

## 5. Results

### 5a. The decisions of the tree

Every node of the flowchart with its question, its answer and the numbers behind it.

In [ ]:
print(open(os.path.join(TREE, 'decision_log.txt')).read())

### 5b. Final result

`conclusion.json` has the final conclusion (from the profile scheme) and the two parts:

- **Bayesian** (`bayesian`): the model chosen by the tree, its ML best fit and the
  PolyChord posterior of all the parameters;
- **profiled** (`profiled`): the final model of the profile scheme, the posterior of r
  from the profile likelihood (with the prior of the tree), the best fit at the minimum
  r = r* and the posterior of the other parameters at r = r*.

In [ ]:
c = json.load(open(os.path.join(TREE, 'conclusion.json')))
b, p = c['bayesian'], c['profiled']
r_true = 0.01113176
print(f"conclusion          : {c['conclusion']}  (true r = {r_true:.4f})")
print(f"model of the tree   : {b['model']}")
print(f"r (tree, Bayesian)  : {b['posterior']['r_tensor']['med']:.4f} "
      f"+- {b['posterior']['r_tensor']['std']:.4f}")
print(f"final model         : {p['model']}  (start: {p['start']}, rejected: {p['rejected']})")
rp = p['r_posterior']
print(f"r (profile)         : {rp['r_med']:.4f} +- {rp['r_std']:.4f}   r* = {p['r_star']:.4f}")

### 5c. Posterior of r: tree vs profile scheme

The Bayesian posterior of r of the model of the tree (PolyChord samples) and the
posterior from the profile likelihood of the final model, $\propto e^{-\Delta\chi^2(r)/2}$
within the prior of the tree.

In [ ]:
from bbsure.config import free_param_names
pc = os.path.join(TREE, f"{b['model']}_PC")
names = free_param_names(yaml.safe_load(open(os.path.join(pc, 'config.yml')))['BBCompSep'])
samp = np.loadtxt(os.path.join(pc, 'polychord', 'pch_equal_weights.txt'))[:, 2:]
r_tree = samp[:, names.index('r_tensor')]

prof = np.load(os.path.join(TREE, 'profile', f"profile_{p['model']}.npz"))
r_grid, chi2 = prof['r'], prof['chi2']
fine = np.linspace(r_grid[0], r_grid[-1], 2000)
from scipy.interpolate import interp1d
d = interp1d(r_grid, chi2, kind='cubic')(fine)
d -= d.min()
post = np.exp(-d / 2)
post /= np.trapezoid(post, fine)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(1e3 * r_tree, 40, density=True, histtype='step', lw=2,
             label=f"tree ({b['model']})")
axes[0].plot(1e3 * fine, post / 1e3, lw=2, label=f"profile ({p['model']})")
axes[1].plot(1e3 * r_grid, chi2 - chi2.min(), 'o-')
axes[1].set_ylabel(r'$\Delta\chi^2$')
axes[1].set_ylim(0, 16)
for ax in axes:
    ax.axvline(1e3 * r_true, color='k', lw=1, label='true r')
    ax.set_xlabel(r'$r\ [10^{-3}]$')
axes[0].legend(); axes[1].set_title('profile likelihood of the final model')
plt.show()

### 5d. Corner plot

`plots/plot_corner.py` overlays the Bayesian posterior of the tree and the posterior of
the profile scheme at r = r*.

In [ ]:
out = os.path.join(WORK, 'corner.png')
subprocess.run(['python', os.path.join(REPO, 'plots', 'plot_corner.py'), TREE,
                os.path.join(TREE, 'profile'), '--r-true', str(r_true),
                '--title', 'example: d12s5, seed 7047', '--out', out], check=True)
from IPython.display import Image
Image(out)

## 6. Next steps

- **Validation on many data sets**: `scripts/make_plan.py` draws seeds, skies and r;
  `nersc/` runs one tree per set on Perlmutter (see `nersc/README.md`) and
  `scripts/collect.py` collects the results; `plots/` makes the tables and figures.
- **Tests** of the logic, without BBCompSep: `python tests/test_decision_tree.py`,
  `python tests/test_profile_scheme.py`.